# LLaMAC EDA

Simple starter notebook for checking downloaded/extracted LLaMAC data.
Run `python scripts/download_llamac.py --prepare` before this notebook.

In [ ]:
# Load basic packages and define project paths used throughout the notebook.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
EXTRACTED_DIR = PROJECT_ROOT / 'data' / 'extracted'
INDEX_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_index.csv'

pd.set_option('display.max_columns', 30)
print(PROJECT_ROOT)

In [ ]:
# Read the prepared file index to confirm extraction succeeded.
index = pd.read_csv(INDEX_PATH)
print(index.shape)
index.head()

In [ ]:
# Summarize how many files exist per participant and by filename type.
display(index.groupby('participant_id').size().describe())
display(index['file_name'].value_counts().head(15))

In [ ]:
# Load all participant answer.csv files into one table for label-level EDA.
answer_rows = []
for row in index[index['file_name'].eq('answer.csv')].itertuples(index=False):
    path = EXTRACTED_DIR / row.relative_path
    df = pd.read_csv(path)
    df.insert(0, 'participant_id', row.participant_id)
    answer_rows.append(df)

answers = pd.concat(answer_rows, ignore_index=True)
print(answers.shape)
answers.head()

In [ ]:
# Check basic missingness and label distributions for the questionnaire columns.
label_cols = ['Valence', 'Arousal', 'Dominance', 'Liking', 'EmotType', 'EmotStr', 'Seen']
display(answers[label_cols].isna().sum())
display(answers[label_cols].describe())

In [ ]:
# Plot compact histograms for the main affect / preference labels.
plot_cols = ['Valence', 'Arousal', 'Dominance', 'Liking', 'EmotType', 'EmotStr']
axes = answers[plot_cols].hist(figsize=(10, 7), bins=5)
plt.suptitle('LLaMAC questionnaire label distributions')
plt.tight_layout()

In [ ]:
# Inspect one biosignal CSV to understand signal columns and sampling density.
sample_signal = index[index['file_name'].str.startswith('band_')].iloc[0]
sample_path = EXTRACTED_DIR / sample_signal.relative_path
signal = pd.read_csv(sample_path)
print(sample_signal.participant_id, sample_signal.file_name, signal.shape)
signal.head()

In [ ]:
# Plot the first trial's GSR, PPG, and skin temperature traces as a quick sanity check.
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=False)
for ax, value_col, time_col in zip(axes, ['GSR', 'PPG', 'SKT'], ['GSR_Time', 'PPG_Time', 'SKT_Time']):
    ax.plot(signal[time_col] - signal[time_col].iloc[0], signal[value_col], linewidth=1)
    ax.set_title(value_col)
    ax.set_xlabel('seconds from trial start')
plt.tight_layout()

In [ ]:
# Build a small trial-level signal summary table for later modeling features.
# This cell samples a few signal files first; increase n_files after confirming runtime.
n_files = 20
summary_rows = []
for row in index[index['file_name'].str.startswith('band_')].head(n_files).itertuples(index=False):
    df = pd.read_csv(EXTRACTED_DIR / row.relative_path)
    summary_rows.append({
        'participant_id': row.participant_id,
        'file_name': row.file_name,
        'n_rows': len(df),
        'GSR_mean': df['GSR'].mean(),
        'PPG_mean': df['PPG'].mean(),
        'SKT_mean': df['SKT'].mean(),
    })
signal_summary = pd.DataFrame(summary_rows)
signal_summary.head()